In [1]:
# Importation des bibliothèques nécessaires
import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')
sns.set(style='whitegrid')

DEPARTMENTS = [61, 69]

# Dictionnaires pour stocker les labels de clusters
cluster_labels = {}
hierarchical_col_name = {}

In [2]:
# Chargement et validation des fichiers de clusters pour chaque département
for dept in DEPARTMENTS:
    cluster_file = f"clusters_DEPT_{dept}.csv"
    if not os.path.exists(cluster_file):
        print(f"Error: {cluster_file} not found")
        continue
    
    try:
        df_clusters = pd.read_csv(cluster_file)
    except Exception as e:
        print(f"Error reading {cluster_file}: {e}")
        continue
    
    # Vérification des colonnes requises
    required_cols = ['station_id', 'cluster_kmeans']
    missing_cols = [col for col in required_cols if col not in df_clusters.columns]
    if missing_cols:
        print(f"Error: {cluster_file} missing required columns: {missing_cols}")
        continue
    
    # Recherche du nom de colonne pour clustering hiérarchique
    hierarchical_col = None
    for possible_name in ['cluster_hierarchical', 'cluster_hc', 'hierarchical_cluster']:
        if possible_name in df_clusters.columns:
            hierarchical_col = possible_name
            break
    
    if hierarchical_col is None:
        print(f"Error: {cluster_file} missing hierarchical clustering column")
        continue
    
    cluster_labels[dept] = df_clusters
    hierarchical_col_name[dept] = hierarchical_col

DEPARTMENTS = list(cluster_labels.keys())
if len(DEPARTMENTS) == 0:
    print("No valid departments found with cluster files. Exiting.")
    exit()

In [3]:
# Construction des dictionnaires de clusters et filtrage des clusters trop petits
def build_clusters(df_clusters, method_col):
    clusters_dict = {}
    for cluster_id in df_clusters[method_col].unique():
        stations = df_clusters[df_clusters[method_col] == cluster_id]['station_id'].astype(str).tolist()
        clusters_dict[cluster_id] = stations
    return clusters_dict

kmeans_clusters = {}
hierarchical_clusters = {}

for dept in DEPARTMENTS:
    df_clusters = cluster_labels[dept]
    
    # Construction des clusters pour K-Means et hiérarchique
    kmeans_clusters[dept] = build_clusters(df_clusters, 'cluster_kmeans')
    hierarchical_clusters[dept] = build_clusters(df_clusters, hierarchical_col_name[dept])
    
    print(f"\nDepartment {dept}:")
    print(f"K-Means clusters: {len(kmeans_clusters[dept])}")
    for cid, stations in kmeans_clusters[dept].items():
        print(f"  Cluster {cid}: {len(stations)} stations")
    
    print(f"\nHierarchical clusters: {len(hierarchical_clusters[dept])}")
    for cid, stations in hierarchical_clusters[dept].items():
        print(f"  Cluster {cid}: {len(stations)} stations")

# Suppression des clusters avec moins de 2 stations
for dept in DEPARTMENTS:
    for method_name, clusters_dict in [("K-Means", kmeans_clusters[dept]), 
                                         ("Hierarchical", hierarchical_clusters[dept])]:
        small_clusters = [cid for cid, stations in clusters_dict.items() if len(stations) < 2]
        if small_clusters:
            for cid in small_clusters:
                del clusters_dict[cid]


Department 61:
K-Means clusters: 7
  Cluster 0: 6 stations
  Cluster 6: 8 stations
  Cluster 1: 1 stations
  Cluster 2: 3 stations
  Cluster 4: 1 stations
  Cluster 5: 1 stations
  Cluster 3: 3 stations

Hierarchical clusters: 7
  Cluster 4: 3 stations
  Cluster 2: 9 stations
  Cluster 6: 1 stations
  Cluster 3: 1 stations
  Cluster 5: 2 stations
  Cluster 1: 3 stations
  Cluster 0: 4 stations

Department 69:
K-Means clusters: 8
  Cluster 1: 6 stations
  Cluster 4: 4 stations
  Cluster 5: 4 stations
  Cluster 0: 4 stations
  Cluster 7: 1 stations
  Cluster 3: 3 stations
  Cluster 2: 1 stations
  Cluster 6: 2 stations

Hierarchical clusters: 8
  Cluster 1: 3 stations
  Cluster 7: 4 stations
  Cluster 4: 4 stations
  Cluster 3: 4 stations
  Cluster 0: 4 stations
  Cluster 5: 1 stations
  Cluster 6: 2 stations
  Cluster 2: 3 stations


In [4]:
# Fonctions de chargement des données et d'ingénierie des caractéristiques temporelles

# Chargement des données d'une station depuis un fichier CSV
def load_station_data(dept, station_id):
    filepath = f"DEPT_{dept}/DEPT_{dept}_STATION_{station_id}.csv"
    if not os.path.exists(filepath):
        return None
    try:
        df = pd.read_csv(filepath, sep=";", parse_dates=["AAAAMMJJHH"], index_col="AAAAMMJJHH")
        if df.empty or 'T' not in df.columns:
            return None
        return df
    except Exception as e:
        print(f"Warning: Failed to load {filepath}: {e}")
        return None

# Ajout de caractéristiques temporelles cycliques et calendaires
def add_time_features(df):
    df = df.copy()
    df['hour'] = df.index.hour
    df['day_of_year'] = df.index.dayofyear
    df['month'] = df.index.month
    df['weekday'] = df.index.weekday
    # Encodage cyclique des heures et jours
    df['sin_hour'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['cos_hour'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['sin_day'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
    df['cos_day'] = np.cos(2 * np.pi * df['day_of_year'] / 365)
    df['is_weekend'] = (df['weekday'] >= 5).astype(int)
    df['hour_squared'] = (df['hour'] / 24) ** 2
    return df

# Ajout de variables de décalage temporel (lag) et moyenne mobile
def add_lag_features(df):
    df = df.copy()
    df['T_lag1'] = df['T'].shift(1)
    df['T_lag24'] = df['T'].shift(24)
    df['T_lag48'] = df['T'].shift(48)
    df['T_rollmean_24'] = df['T'].rolling(window=24).mean()
    df.dropna(inplace=True)
    return df

In [5]:
# Fonctions de division temporelle des données et création de séquences pour le modèle

# Division des données en ensembles train/test selon des ratios temporels
def time_split(df, train_ratio=0.80):
    n_total = len(df)
    n_train = int(n_total * train_ratio)
    
    train = df.iloc[:n_train]
    test = df.iloc[n_train:]
    
    engineered_features = ['hour', 'month', 'weekday', 'sin_hour', 'cos_hour',
                          'sin_day', 'cos_day', 'is_weekend', 'hour_squared',
                          'T_lag1', 'T_lag24', 'T_lag48']
    feature_cols = [c for c in engineered_features if c in df.columns]
    
    X_train = train[feature_cols]
    X_test = test[feature_cols]
    
    y_train = train['T_rollmean_24']
    y_test = test['T_rollmean_24']
    
    if X_train.isnull().any().any() or X_test.isnull().any().any():
        raise ValueError("NaN values detected in features after engineering!")
    if y_train.isnull().any() or y_test.isnull().any():
        raise ValueError("NaN values detected in target variable!")
    
    return X_train, X_test, y_train, y_test

# Création de séquences temporelles avec fenêtre glissante pour l'entrée du modèle
def create_sequences(X, y, window_size=24):
    X_seq, y_seq = [], []
    
    if window_size < 24:
        raise ValueError(f"window_size must be >= 24 for diverse lag features, got {window_size}")
    
    for i in range(len(X) - window_size):
        window_data = X[i:i+window_size]
        
        # Extraction de points temporels clés de la fenêtre
        features = []
        features.append(window_data[-1])
        features.append(window_data[-3])
        features.append(window_data[-6])
        features.append(window_data[-12])
        features.append(window_data[-24])
        features.append(window_data[0])
        
        X_seq.append(np.concatenate(features))
        y_seq.append(y[i+window_size])
    
    return np.array(X_seq), np.array(y_seq)

In [6]:
# Chargement et préparation des données pour toutes les stations de chaque département

station_raw_data_all = {}

for dept in DEPARTMENTS:
    station_raw_data_all[dept] = {}
    all_stations = cluster_labels[dept]['station_id'].astype(str).unique()
    
    skipped_initial = 0
    
    # Boucle de chargement pour chaque station
    for station_id in all_stations:
        df = load_station_data(dept, station_id)
        if df is None or len(df) < 100:
            skipped_initial += 1
            continue
        
        df = add_time_features(df)
        df = add_lag_features(df)
        
        if len(df) < 200:
            skipped_initial += 1
            continue
        
        try:
            X_train, X_test, y_train, y_test = time_split(df)
        except ValueError as e:
            print(f"  Skipping station {station_id}: {e}")
            skipped_initial += 1
            continue
        
        # Stockage des ensembles train/test pour la station
        station_raw_data_all[dept][station_id] = {
            'X_train': X_train,
            'X_test': X_test,
            'y_train': y_train,
            'y_test': y_test
        }
    
    print(f"Department {dept}: Loaded {len(station_raw_data_all[dept])} stations (skipped {skipped_initial} due to insufficient data)")

Department 61: Loaded 23 stations (skipped 0 due to insufficient data)
Department 69: Loaded 25 stations (skipped 0 due to insufficient data)


In [7]:
# Fonctions d'entraînement local pour l'apprentissage fédéré (FedAvg et FedProx)

# Évaluation du modèle de régression linéaire
def eval_lr(w, b, X):
    return X @ w + b

# Entraînement local avec FedAvg (descente de gradient stochastique)
def local_train_fedavg_sgd(X, y, w_global, b_global, learning_rate=0.005, epochs=15, l2_lambda=0.001):
    n, d = X.shape
    if n == 0:
        return w_global, b_global
    
    w_local = w_global.copy()
    b_local = b_global
    
    for epoch in range(epochs):
        y_pred = X @ w_local + b_local
        error = y_pred - y
        
        grad_w = (2/n) * (X.T @ error) + l2_lambda * w_local
        grad_b = (2/n) * np.sum(error)
        
        w_local = w_local - learning_rate * grad_w
        b_local = b_local - learning_rate * grad_b
    
    return w_local, b_local

# Entraînement local avec FedProx (ajout d'un terme proximal pour la régularisation)
def local_train_fedprox_sgd(X, y, w_global, b_global, mu=0.05, learning_rate=0.005, epochs=15, l2_lambda=0.001):
    n, d = X.shape
    if n == 0:
        return w_global, b_global
    
    w_local = w_global.copy()
    b_local = b_global
    
    for epoch in range(epochs):
        y_pred = X @ w_local + b_local
        error = y_pred - y
        
        # Gradient avec terme proximal mu*(w_local - w_global)
        grad_w = (2/n) * (X.T @ error) + mu * (w_local - w_global) + l2_lambda * w_local
        grad_b = (2/n) * np.sum(error)
        
        w_local = w_local - learning_rate * grad_w
        b_local = b_local - learning_rate * grad_b
    
    return w_local, b_local

In [8]:
# Agrégation des poids et biais locaux par moyenne pondérée selon la taille des ensembles d'entraînement

def aggregate_weights(weights_list, bias_list, train_sizes):
    total = sum(train_sizes)
    if total == 0:
        return weights_list[0], bias_list[0]
    
    # Moyenne pondérée des poids par le nombre d'échantillons de chaque client
    w_agg = sum(w * n for w, n in zip(weights_list, train_sizes)) / total
    
    b_agg = sum(b * n for b, n in zip(bias_list, train_sizes)) / total
    
    return w_agg, b_agg

In [9]:
# Préparation des clients d'un cluster : normalisation des données et création de séquences

def _prepare_clients_for_cluster(cluster_stations, dept, window_size=24):
    valid_stations = []
    for sid in cluster_stations:
        if sid in station_raw_data_all[dept]:
            valid_stations.append(sid)

    clients = []
    total_train_samples = 0
    skipped_in_cluster = 0

    for sid in valid_stations:
        raw_data = station_raw_data_all[dept][sid]

        # Normalisation locale des features et de la variable cible
        local_X_scaler = StandardScaler()
        local_y_scaler = StandardScaler()

        X_train_scaled = local_X_scaler.fit_transform(raw_data['X_train'])
        X_test_scaled = local_X_scaler.transform(raw_data['X_test'])

        y_train_scaled = local_y_scaler.fit_transform(raw_data['y_train'].values.reshape(-1, 1)).flatten()
        y_test_scaled = local_y_scaler.transform(raw_data['y_test'].values.reshape(-1, 1)).flatten()

        if len(X_train_scaled) <= window_size or len(X_test_scaled) <= window_size:
            skipped_in_cluster += 1
            continue

        # Création de séquences temporelles pour chaque ensemble
        X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train_scaled, window_size)
        X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test_scaled, window_size)

        clients.append({
            'station_id': sid,
            'X_train': X_train_seq,
            'X_test': X_test_seq,
            'y_train': y_train_seq,
            'y_test': y_test_seq,
            'n_train': len(X_train_seq),
            'y_scaler': local_y_scaler,
            'X_scaler': local_X_scaler
        })
        total_train_samples += len(X_train_seq)

    return clients, total_train_samples, skipped_in_cluster

In [10]:
# Entraînement du modèle global par apprentissage fédéré sur plusieurs rounds

def _train_global_lr_model(clients, num_rounds=5, mu=0.0, use_fedprox=False):
    n_features = clients[0]['X_train'].shape[1]
    w_global = np.zeros(n_features)
    b_global = 0.0

    round_metrics = []

    for round_idx in range(num_rounds):
        local_weights = []
        local_biases = []
        train_sizes = []

        # Entraînement local sur chaque client
        for client in clients:
            X_train = client['X_train']
            y_train = client['y_train']

            if use_fedprox:
                w_local, b_local = local_train_fedprox_sgd(X_train, y_train, w_global, b_global, mu)
            else:
                w_local, b_local = local_train_fedavg_sgd(X_train, y_train, w_global, b_global)

            local_weights.append(w_local)
            local_biases.append(b_local)
            train_sizes.append(client['n_train'])

        # Agrégation des modèles locaux en modèle global
        w_global, b_global = aggregate_weights(local_weights, local_biases, train_sizes)

        # Calcul de la perte d'entraînement moyenne
        train_loss_sum = 0
        total_samples = 0
        for client in clients:
            y_pred_scaled = eval_lr(w_global, b_global, client['X_train'])
            train_loss = mean_squared_error(client['y_train'], y_pred_scaled)
            train_loss_sum += train_loss * len(client['y_train'])
            total_samples += len(client['y_train'])

        avg_train_loss = train_loss_sum / total_samples if total_samples > 0 else 0
        round_metrics.append({'round': round_idx + 1, 'train_loss': avg_train_loss})

    return w_global, b_global, round_metrics

In [11]:
# Évaluation du modèle global sur les données de test de chaque client avec dénormalisation

def _evaluate_global_lr_model(clients, w_global, b_global):
    results = []
    client_scalers = {}

    for client in clients:
        X_test = client['X_test']
        y_test_scaled = client['y_test']

        # Prédiction avec le modèle global
        y_pred_scaled = eval_lr(w_global, b_global, X_test)

        # Dénormalisation des prédictions et des vraies valeurs
        y_test = client['y_scaler'].inverse_transform(y_test_scaled.reshape(-1, 1)).flatten()
        y_pred = client['y_scaler'].inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()

        # Calcul des métriques de performance
        mse = mean_squared_error(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        results.append({
            'station_id': client['station_id'],
            'mse': mse,
            'mae': mae,
            'r2': r2
        })

        client_scalers[client['station_id']] = {
            'X_scaler': client['X_scaler'],
            'y_scaler': client['y_scaler']
        }

    return results, client_scalers

In [12]:
# Fonction principale pour exécuter l'apprentissage fédéré sur un cluster de stations

def run_fl_for_cluster(cluster_stations, dept, num_rounds=5, mu=0.0, use_fedprox=False):
    if dept not in station_raw_data_all:
        raise KeyError(f"Department {dept} not found in station_raw_data_all")

    # Préparation des clients du cluster
    clients, total_train_samples, _skipped_in_cluster = _prepare_clients_for_cluster(
        cluster_stations, dept, window_size=24
    )

    if len(clients) == 0:
        print(f"    Warning: No valid clients after sequence creation. Skipping.")
        return None

    if len(clients) < 2:
        print(f"    Warning: Only {len(clients)} client(s) in cluster. FL requires ≥2. Skipping.")
        return None

    MIN_SAMPLES_PER_CLUSTER = 100
    if total_train_samples < MIN_SAMPLES_PER_CLUSTER:
        print(f"    Warning: Only {total_train_samples} training samples (<{MIN_SAMPLES_PER_CLUSTER}). Skipping for stability.")
        return None

    # Entraînement du modèle global par FL
    w_global, b_global, round_metrics = _train_global_lr_model(
        clients, num_rounds=num_rounds, mu=mu, use_fedprox=use_fedprox
    )
    
    # Évaluation sur les données de test
    results, client_scalers = _evaluate_global_lr_model(clients, w_global, b_global)

    return {
        'w': w_global,
        'b': b_global,
        'results': results,
        'round_metrics': round_metrics,
        'client_scalers': client_scalers
    }

In [13]:
# Exécution de CFL-1 : apprentissage fédéré avec clustering K-Means et algorithme FedAvg

CFL1_results = {}
os.makedirs('CFL_models', exist_ok=True)

for dept in DEPARTMENTS:
    print(f"\nDepartment {dept} - CFL-1 (K-Means + FedAvg)")
    CFL1_results[dept] = {}
    
    # Entraînement d'un modèle pour chaque cluster K-Means
    for cluster_id, cluster_stations in kmeans_clusters[dept].items():
        print(f"  Cluster {cluster_id}: {len(cluster_stations)} stations")
        
        result = run_fl_for_cluster(cluster_stations, dept, num_rounds=20, use_fedprox=False)
        
        if result is not None:
            CFL1_results[dept][cluster_id] = result
            df_res = pd.DataFrame(result['results'])
            print(f"    Avg MSE: {df_res['mse'].mean():.4f}, Avg MAE: {df_res['mae'].mean():.4f}, Avg R²: {df_res['r2'].mean():.4f}")
            
            # Sauvegarde du modèle entraîné
            model_path = f"CFL_models/CFL1_DEPT_{dept}_CLUSTER_{cluster_id}.pkl"
            import pickle
            with open(model_path, 'wb') as f:
                pickle.dump({
                    'w': result['w'], 
                    'b': result['b'],
                    'client_scalers': result.get('client_scalers', {})
                }, f)
            print(f"    Model saved: {model_path}")


Department 61 - CFL-1 (K-Means + FedAvg)
  Cluster 0: 6 stations
    Avg MSE: 0.6920, Avg MAE: 0.6466, Avg R²: 0.9811
    Model saved: CFL_models/CFL1_DEPT_61_CLUSTER_0.pkl
  Cluster 6: 8 stations
    Avg MSE: 0.6103, Avg MAE: 0.6079, Avg R²: 0.9822
    Model saved: CFL_models/CFL1_DEPT_61_CLUSTER_6.pkl
  Cluster 2: 3 stations
    Avg MSE: 0.5981, Avg MAE: 0.5972, Avg R²: 0.9832
    Model saved: CFL_models/CFL1_DEPT_61_CLUSTER_2.pkl
  Cluster 3: 3 stations
    Avg MSE: 0.8229, Avg MAE: 0.7052, Avg R²: 0.9760
    Model saved: CFL_models/CFL1_DEPT_61_CLUSTER_3.pkl

Department 69 - CFL-1 (K-Means + FedAvg)
  Cluster 1: 6 stations
    Avg MSE: 0.9353, Avg MAE: 0.7486, Avg R²: 0.9831
    Model saved: CFL_models/CFL1_DEPT_69_CLUSTER_1.pkl
  Cluster 4: 4 stations
    Avg MSE: 0.7124, Avg MAE: 0.6565, Avg R²: 0.9863
    Model saved: CFL_models/CFL1_DEPT_69_CLUSTER_4.pkl
  Cluster 5: 4 stations
    Avg MSE: 0.9168, Avg MAE: 0.7436, Avg R²: 0.9834
    Model saved: CFL_models/CFL1_DEPT_69_CLUSTE

In [14]:
# Compilation et analyse des résultats CFL-1 par département avec moyennes pondérées

all_cfl1_results = []
for dept in DEPARTMENTS:
    for cluster_id, result in CFL1_results[dept].items():
        df_res = pd.DataFrame(result['results'])
        df_res['dept'] = dept
        df_res['cluster_id'] = cluster_id
        df_res['method'] = 'CFL-1'
        all_cfl1_results.append(df_res)

if all_cfl1_results:
    df_cfl1 = pd.concat(all_cfl1_results, ignore_index=True)
else:
    df_cfl1 = pd.DataFrame(columns=['station_id', 'mse', 'mae', 'r2', 'dept', 'cluster_id', 'method'])

# Calcul des moyennes pondérées par cluster pour chaque département
for dept in DEPARTMENTS:
    df_cfl1_dept = df_cfl1[df_cfl1['dept'] == dept]
    cluster_stats = df_cfl1_dept.groupby('cluster_id').agg({
        'mse': 'mean',
        'mae': 'mean',
        'r2': 'mean',
        'station_id': 'count'
    })
    cluster_stats.columns = ['mse', 'mae', 'r2', 'n_stations']
    total_stations = cluster_stats['n_stations'].sum()
    mse_weighted = (cluster_stats['mse'] * cluster_stats['n_stations']).sum() / total_stations
    mae_weighted = (cluster_stats['mae'] * cluster_stats['n_stations']).sum() / total_stations
    r2_weighted = (cluster_stats['r2'] * cluster_stats['n_stations']).sum() / total_stations
    print(f"\nCFL-1 DEPT {dept} Cluster-Weighted Avg - MSE: {mse_weighted:.4f}, MAE: {mae_weighted:.4f}, R²: {r2_weighted:.4f}")


CFL-1 DEPT 61 Cluster-Weighted Avg - MSE: 0.6649, MAE: 0.6325, R²: 0.9811

CFL-1 DEPT 69 Cluster-Weighted Avg - MSE: 0.8540, MAE: 0.7169, R²: 0.9842


In [15]:
# Exécution de CFL-2 : apprentissage fédéré avec clustering K-Means et algorithme FedProx

CFL2_results = {}
FEDPROX_MU = 0.05

for dept in DEPARTMENTS:
    print(f"\nDepartment {dept} - CFL-2 (K-Means + FedProx, mu={FEDPROX_MU})")
    CFL2_results[dept] = {}
    
    # Entraînement avec FedProx (régularisation proximale mu=0.05)
    for cluster_id, cluster_stations in kmeans_clusters[dept].items():
        print(f"  Cluster {cluster_id}: {len(cluster_stations)} stations")
        
        result = run_fl_for_cluster(cluster_stations, dept, num_rounds=20, mu=FEDPROX_MU, use_fedprox=True)
        
        if result is not None:
            CFL2_results[dept][cluster_id] = result
            df_res = pd.DataFrame(result['results'])
            print(f"    Avg MSE: {df_res['mse'].mean():.4f}, Avg MAE: {df_res['mae'].mean():.4f}, Avg R²: {df_res['r2'].mean():.4f}")
            
            # Sauvegarde du modèle CFL-2
            model_path = f"CFL_models/CFL2_DEPT_{dept}_CLUSTER_{cluster_id}.pkl"
            import pickle
            with open(model_path, 'wb') as f:
                pickle.dump({
                    'w': result['w'], 
                    'b': result['b'],
                    'client_scalers': result.get('client_scalers', {})
                }, f)
            print(f"    Model saved: {model_path}")


Department 61 - CFL-2 (K-Means + FedProx, mu=0.05)
  Cluster 0: 6 stations
    Avg MSE: 0.6928, Avg MAE: 0.6470, Avg R²: 0.9811
    Model saved: CFL_models/CFL2_DEPT_61_CLUSTER_0.pkl
  Cluster 6: 8 stations
    Avg MSE: 0.6110, Avg MAE: 0.6082, Avg R²: 0.9822
    Model saved: CFL_models/CFL2_DEPT_61_CLUSTER_6.pkl
  Cluster 2: 3 stations
    Avg MSE: 0.5988, Avg MAE: 0.5975, Avg R²: 0.9832
    Model saved: CFL_models/CFL2_DEPT_61_CLUSTER_2.pkl
  Cluster 3: 3 stations
    Avg MSE: 0.8237, Avg MAE: 0.7055, Avg R²: 0.9760
    Model saved: CFL_models/CFL2_DEPT_61_CLUSTER_3.pkl

Department 69 - CFL-2 (K-Means + FedProx, mu=0.05)
  Cluster 1: 6 stations
    Avg MSE: 0.9363, Avg MAE: 0.7490, Avg R²: 0.9831
    Model saved: CFL_models/CFL2_DEPT_69_CLUSTER_1.pkl
  Cluster 4: 4 stations
    Avg MSE: 0.7134, Avg MAE: 0.6570, Avg R²: 0.9863
    Model saved: CFL_models/CFL2_DEPT_69_CLUSTER_4.pkl
  Cluster 5: 4 stations
    Avg MSE: 0.9179, Avg MAE: 0.7440, Avg R²: 0.9834
    Model saved: CFL_models

In [16]:
# Compilation et analyse des résultats CFL-2 avec moyennes pondérées par département

all_cfl2_results = []
for dept in DEPARTMENTS:
    for cluster_id, result in CFL2_results[dept].items():
        df_res = pd.DataFrame(result['results'])
        df_res['dept'] = dept
        df_res['cluster_id'] = cluster_id
        df_res['method'] = 'CFL-2'
        all_cfl2_results.append(df_res)

if all_cfl2_results:
    df_cfl2 = pd.concat(all_cfl2_results, ignore_index=True)
else:
    df_cfl2 = pd.DataFrame(columns=['station_id', 'mse', 'mae', 'r2', 'dept', 'cluster_id', 'method'])

# Calcul des statistiques pondérées par le nombre de stations dans chaque cluster
for dept in DEPARTMENTS:
    df_cfl2_dept = df_cfl2[df_cfl2['dept'] == dept]
    cluster_stats = df_cfl2_dept.groupby('cluster_id').agg({
        'mse': 'mean',
        'mae': 'mean',
        'r2': 'mean',
        'station_id': 'count'
    })
    cluster_stats.columns = ['mse', 'mae', 'r2', 'n_stations']
    total_stations = cluster_stats['n_stations'].sum()
    mse_weighted = (cluster_stats['mse'] * cluster_stats['n_stations']).sum() / total_stations
    mae_weighted = (cluster_stats['mae'] * cluster_stats['n_stations']).sum() / total_stations
    r2_weighted = (cluster_stats['r2'] * cluster_stats['n_stations']).sum() / total_stations
    print(f"\nCFL-2 DEPT {dept} Cluster-Weighted Avg - MSE: {mse_weighted:.4f}, MAE: {mae_weighted:.4f}, R²: {r2_weighted:.4f}")


CFL-2 DEPT 61 Cluster-Weighted Avg - MSE: 0.6656, MAE: 0.6328, R²: 0.9811

CFL-2 DEPT 69 Cluster-Weighted Avg - MSE: 0.8550, MAE: 0.7173, R²: 0.9841


In [17]:
# Exécution de CFL-3 : apprentissage fédéré avec clustering hiérarchique et algorithme FedAvg

CFL3_results = {}

for dept in DEPARTMENTS:
    print(f"\nDepartment {dept} - CFL-3 (Hierarchical + FedAvg)")
    CFL3_results[dept] = {}
    
    # Entraînement d'un modèle pour chaque cluster hiérarchique avec FedAvg
    for cluster_id, cluster_stations in hierarchical_clusters[dept].items():
        print(f"  Cluster {cluster_id}: {len(cluster_stations)} stations")
        
        result = run_fl_for_cluster(cluster_stations, dept, num_rounds=20, use_fedprox=False)
        
        if result is not None:
            CFL3_results[dept][cluster_id] = result
            df_res = pd.DataFrame(result['results'])
            print(f"    Avg MSE: {df_res['mse'].mean():.4f}, Avg MAE: {df_res['mae'].mean():.4f}, Avg R²: {df_res['r2'].mean():.4f}")
            
            # Sauvegarde du modèle CFL-3
            model_path = f"CFL_models/CFL3_DEPT_{dept}_CLUSTER_{cluster_id}.pkl"
            import pickle
            with open(model_path, 'wb') as f:
                pickle.dump({
                    'w': result['w'], 
                    'b': result['b'],
                    'client_scalers': result.get('client_scalers', {})
                }, f)
            print(f"    Model saved: {model_path}")


Department 61 - CFL-3 (Hierarchical + FedAvg)
  Cluster 4: 3 stations
    Avg MSE: 0.6433, Avg MAE: 0.6265, Avg R²: 0.9829
    Model saved: CFL_models/CFL3_DEPT_61_CLUSTER_4.pkl
  Cluster 2: 9 stations
    Avg MSE: 0.6599, Avg MAE: 0.6296, Avg R²: 0.9807
    Model saved: CFL_models/CFL3_DEPT_61_CLUSTER_2.pkl
  Cluster 5: 2 stations
    Avg MSE: 0.7385, Avg MAE: 0.6611, Avg R²: 0.9817
    Model saved: CFL_models/CFL3_DEPT_61_CLUSTER_5.pkl
  Cluster 1: 3 stations
    Avg MSE: 0.5981, Avg MAE: 0.5972, Avg R²: 0.9832
    Model saved: CFL_models/CFL3_DEPT_61_CLUSTER_1.pkl
  Cluster 0: 4 stations
    Avg MSE: 0.8227, Avg MAE: 0.7064, Avg R²: 0.9773
    Model saved: CFL_models/CFL3_DEPT_61_CLUSTER_0.pkl

Department 69 - CFL-3 (Hierarchical + FedAvg)
  Cluster 1: 3 stations
    Avg MSE: 0.7674, Avg MAE: 0.6814, Avg R²: 0.9861
    Model saved: CFL_models/CFL3_DEPT_69_CLUSTER_1.pkl
  Cluster 7: 4 stations
    Avg MSE: 0.7124, Avg MAE: 0.6565, Avg R²: 0.9863
    Model saved: CFL_models/CFL3_DEPT

In [18]:
# Compilation et analyse des résultats CFL-3 avec moyennes pondérées

all_cfl3_results = []
for dept in DEPARTMENTS:
    for cluster_id, result in CFL3_results[dept].items():
        df_res = pd.DataFrame(result['results'])
        df_res['dept'] = dept
        df_res['cluster_id'] = cluster_id
        df_res['method'] = 'CFL-3'
        all_cfl3_results.append(df_res)

if all_cfl3_results:
    df_cfl3 = pd.concat(all_cfl3_results, ignore_index=True)
else:
    df_cfl3 = pd.DataFrame(columns=['station_id', 'mse', 'mae', 'r2', 'dept', 'cluster_id', 'method'])

# Calcul des moyennes pondérées par cluster pour CFL-3
for dept in DEPARTMENTS:
    df_cfl3_dept = df_cfl3[df_cfl3['dept'] == dept]
    cluster_stats = df_cfl3_dept.groupby('cluster_id').agg({
        'mse': 'mean',
        'mae': 'mean',
        'r2': 'mean',
        'station_id': 'count'
    })
    cluster_stats.columns = ['mse', 'mae', 'r2', 'n_stations']
    total_stations = cluster_stats['n_stations'].sum()
    mse_weighted = (cluster_stats['mse'] * cluster_stats['n_stations']).sum() / total_stations
    mae_weighted = (cluster_stats['mae'] * cluster_stats['n_stations']).sum() / total_stations
    r2_weighted = (cluster_stats['r2'] * cluster_stats['n_stations']).sum() / total_stations
    print(f"\nCFL-3 DEPT {dept} Cluster-Weighted Avg - MSE: {mse_weighted:.4f}, MAE: {mae_weighted:.4f}, R²: {r2_weighted:.4f}")


CFL-3 DEPT 61 Cluster-Weighted Avg - MSE: 0.6872, MAE: 0.6422, R²: 0.9808

CFL-3 DEPT 69 Cluster-Weighted Avg - MSE: 0.8464, MAE: 0.7150, R²: 0.9843


In [19]:
# Exécution de CFL-4 : apprentissage fédéré avec clustering hiérarchique et algorithme FedProx

CFL4_results = {}

for dept in DEPARTMENTS:
    print(f"\nDepartment {dept} - CFL-4 (Hierarchical + FedProx, mu={FEDPROX_MU})")
    CFL4_results[dept] = {}
    
    # Entraînement avec clustering hiérarchique et FedProx (régularisation proximale)
    for cluster_id, cluster_stations in hierarchical_clusters[dept].items():
        print(f"  Cluster {cluster_id}: {len(cluster_stations)} stations")
        
        result = run_fl_for_cluster(cluster_stations, dept, num_rounds=20, mu=FEDPROX_MU, use_fedprox=True)
        
        if result is not None:
            CFL4_results[dept][cluster_id] = result
            df_res = pd.DataFrame(result['results'])
            print(f"    Avg MSE: {df_res['mse'].mean():.4f}, Avg MAE: {df_res['mae'].mean():.4f}, Avg R²: {df_res['r2'].mean():.4f}")
            
            # Sauvegarde du modèle CFL-4
            model_path = f"CFL_models/CFL4_DEPT_{dept}_CLUSTER_{cluster_id}.pkl"
            import pickle
            with open(model_path, 'wb') as f:
                pickle.dump({
                    'w': result['w'], 
                    'b': result['b'],
                    'client_scalers': result.get('client_scalers', {})
                }, f)
            print(f"    Model saved: {model_path}")


Department 61 - CFL-4 (Hierarchical + FedProx, mu=0.05)
  Cluster 4: 3 stations
    Avg MSE: 0.6441, Avg MAE: 0.6269, Avg R²: 0.9829
    Model saved: CFL_models/CFL4_DEPT_61_CLUSTER_4.pkl
  Cluster 2: 9 stations
    Avg MSE: 0.6607, Avg MAE: 0.6300, Avg R²: 0.9806
    Model saved: CFL_models/CFL4_DEPT_61_CLUSTER_2.pkl
  Cluster 5: 2 stations
    Avg MSE: 0.7393, Avg MAE: 0.6615, Avg R²: 0.9816
    Model saved: CFL_models/CFL4_DEPT_61_CLUSTER_5.pkl
  Cluster 1: 3 stations
    Avg MSE: 0.5988, Avg MAE: 0.5975, Avg R²: 0.9832
    Model saved: CFL_models/CFL4_DEPT_61_CLUSTER_1.pkl
  Cluster 0: 4 stations
    Avg MSE: 0.8235, Avg MAE: 0.7067, Avg R²: 0.9773
    Model saved: CFL_models/CFL4_DEPT_61_CLUSTER_0.pkl

Department 69 - CFL-4 (Hierarchical + FedProx, mu=0.05)
  Cluster 1: 3 stations
    Avg MSE: 0.7683, Avg MAE: 0.6818, Avg R²: 0.9861
    Model saved: CFL_models/CFL4_DEPT_69_CLUSTER_1.pkl
  Cluster 7: 4 stations
    Avg MSE: 0.7134, Avg MAE: 0.6570, Avg R²: 0.9863
    Model saved: 

In [20]:
# Compilation et analyse des résultats CFL-4 avec moyennes pondérées

all_cfl4_results = []
for dept in DEPARTMENTS:
    for cluster_id, result in CFL4_results[dept].items():
        df_res = pd.DataFrame(result['results'])
        df_res['dept'] = dept
        df_res['cluster_id'] = cluster_id
        df_res['method'] = 'CFL-4'
        all_cfl4_results.append(df_res)

if all_cfl4_results:
    df_cfl4 = pd.concat(all_cfl4_results, ignore_index=True)
else:
    df_cfl4 = pd.DataFrame(columns=['station_id', 'mse', 'mae', 'r2', 'dept', 'cluster_id', 'method'])

# Affichage des statistiques pondérées pour CFL-4
for dept in DEPARTMENTS:
    df_cfl4_dept = df_cfl4[df_cfl4['dept'] == dept]
    cluster_stats = df_cfl4_dept.groupby('cluster_id').agg({
        'mse': 'mean',
        'mae': 'mean',
        'r2': 'mean',
        'station_id': 'count'
    })
    cluster_stats.columns = ['mse', 'mae', 'r2', 'n_stations']
    total_stations = cluster_stats['n_stations'].sum()
    mse_weighted = (cluster_stats['mse'] * cluster_stats['n_stations']).sum() / total_stations
    mae_weighted = (cluster_stats['mae'] * cluster_stats['n_stations']).sum() / total_stations
    r2_weighted = (cluster_stats['r2'] * cluster_stats['n_stations']).sum() / total_stations
    print(f"\nCFL-4 DEPT {dept} Cluster-Weighted Avg - MSE: {mse_weighted:.4f}, MAE: {mae_weighted:.4f}, R²: {r2_weighted:.4f}")


CFL-4 DEPT 61 Cluster-Weighted Avg - MSE: 0.6880, MAE: 0.6425, R²: 0.9808

CFL-4 DEPT 69 Cluster-Weighted Avg - MSE: 0.8474, MAE: 0.7154, R²: 0.9843


In [21]:
# Consolidation de tous les résultats CFL et sauvegarde des fichiers détaillés par département

df_all = pd.concat([df_cfl1, df_cfl2, df_cfl3, df_cfl4], ignore_index=True)

# Sauvegarde des résultats détaillés pour chaque département
for dept in DEPARTMENTS:
    df_dept_detailed = df_all[df_all['dept'] == dept]
    detailed_csv_path = f'CFL_Detailed_Results_DEPT_{dept}.csv'
    df_dept_detailed.to_csv(detailed_csv_path, index=False)
    print(f"Saved: {detailed_csv_path} ({len(df_dept_detailed)} stations)")

Saved: CFL_Detailed_Results_DEPT_61.csv (82 stations)
Saved: CFL_Detailed_Results_DEPT_69.csv (94 stations)


In [22]:
# Génération des tableaux récapitulatifs par département et sauvegarde en CSV

for dept in DEPARTMENTS:
    output_dir = f'DEPT_{dept}_OUTPUTS'
    os.makedirs(output_dir, exist_ok=True)
    
    df_dept = df_all[df_all['dept'] == dept]
    
    # Agrégation des métriques moyennes par méthode CFL
    summary_dept = df_dept.groupby('method').agg({
        'mse': 'mean',
        'mae': 'mean',
        'r2': 'mean'
    }).reset_index()
    
    summary_dept.columns = ['Method', 'MSE', 'MAE', 'R²']
    summary_dept['Clustering'] = ['K-means', 'K-means', 'Hierarchical', 'Hierarchical']
    summary_dept['FL'] = ['FedAvg', 'FedProx', 'FedAvg', 'FedProx']
    summary_dept = summary_dept[['Method', 'Clustering', 'FL', 'MSE', 'MAE', 'R²']]
    
    print(f"\nDEPARTMENT {dept} RESULTS:")
    print("-" * 80)
    print(summary_dept.to_string(index=False))
    print("-" * 80)
    
    summary_dept.to_csv(f'CFL_Summary_DEPT_{dept}.csv', index=False)
    print(f"Saved to CFL_Summary_DEPT_{dept}.csv")


DEPARTMENT 61 RESULTS:
--------------------------------------------------------------------------------
Method   Clustering      FL      MSE      MAE       R²
 CFL-1      K-means  FedAvg 0.664868 0.632468 0.981072
 CFL-2      K-means FedProx 0.665627 0.632829 0.981050
 CFL-3 Hierarchical  FedAvg 0.687204 0.642164 0.980800
 CFL-4 Hierarchical FedProx 0.687980 0.642528 0.980778
--------------------------------------------------------------------------------
Saved to CFL_Summary_DEPT_61.csv

DEPARTMENT 69 RESULTS:
--------------------------------------------------------------------------------
Method   Clustering      FL      MSE      MAE       R²
 CFL-1      K-means  FedAvg 0.854012 0.716867 0.984153
 CFL-2      K-means FedProx 0.855039 0.717305 0.984134
 CFL-3 Hierarchical  FedAvg 0.846387 0.714961 0.984326
 CFL-4 Hierarchical FedProx 0.847422 0.715403 0.984307
--------------------------------------------------------------------------------
Saved to CFL_Summary_DEPT_69.csv


In [23]:
# Génération de graphiques à barres comparant les MSE des 4 méthodes CFL par département

for dept in DEPARTMENTS:
    output_dir = f'DEPT_{dept}_OUTPUTS'
    os.makedirs(output_dir, exist_ok=True)
    
    df_dept = df_all[df_all['dept'] == dept]
    summary_dept = df_dept.groupby('method').agg({'mse': 'mean', 'mae': 'mean', 'r2': 'mean'}).reset_index()
    summary_dept.columns = ['Method', 'MSE', 'MAE', 'R²']
    
    x_pos = np.arange(len(summary_dept))
    colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']
    
    # Création du graphique de comparaison MSE
    plt.figure(figsize=(10, 6))
    plt.bar(x_pos, summary_dept['MSE'], color=colors, alpha=0.8, edgecolor='black')
    plt.xlabel('Method', fontsize=12)
    plt.ylabel('Mean Squared Error (MSE)', fontsize=12)
    plt.title(f'DEPT {dept} - CFL Methods Comparison (MSE)', fontsize=14, fontweight='bold')
    plt.xticks(x_pos, summary_dept['Method'], rotation=0)
    plt.grid(axis='y', alpha=0.3)
    for i, v in enumerate(summary_dept['MSE']):
        plt.text(i, v + 0.01, f"{v:.4f}", ha='center', va='bottom', fontsize=10)
    plt.tight_layout()
    plt.savefig(f'{output_dir}/CFL_MSE_Comparison_DEPT_{dept}.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {output_dir}/CFL_MSE_Comparison_DEPT_{dept}.png")

Saved: DEPT_61_OUTPUTS/CFL_MSE_Comparison_DEPT_61.png
Saved: DEPT_69_OUTPUTS/CFL_MSE_Comparison_DEPT_69.png


In [24]:
# Génération de graphiques à barres comparant les MAE des 4 méthodes CFL par département

for dept in DEPARTMENTS:
    output_dir = f'DEPT_{dept}_OUTPUTS'
    os.makedirs(output_dir, exist_ok=True)
    
    df_dept = df_all[df_all['dept'] == dept]
    summary_dept = df_dept.groupby('method').agg({'mse': 'mean', 'mae': 'mean', 'r2': 'mean'}).reset_index()
    summary_dept.columns = ['Method', 'MSE', 'MAE', 'R²']
    
    x_pos = np.arange(len(summary_dept))
    colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']
    
    # Visualisation de la comparaison MAE entre méthodes
    plt.figure(figsize=(10, 6))
    plt.bar(x_pos, summary_dept['MAE'], color=colors, alpha=0.8, edgecolor='black')
    plt.xlabel('Method', fontsize=12)
    plt.ylabel('Mean Absolute Error (MAE)', fontsize=12)
    plt.title(f'DEPT {dept} - CFL Methods Comparison (MAE)', fontsize=14, fontweight='bold')
    plt.xticks(x_pos, summary_dept['Method'], rotation=0)
    plt.grid(axis='y', alpha=0.3)
    for i, v in enumerate(summary_dept['MAE']):
        plt.text(i, v + 0.01, f"{v:.4f}", ha='center', va='bottom', fontsize=10)
    plt.tight_layout()
    plt.savefig(f'{output_dir}/CFL_MAE_Comparison_DEPT_{dept}.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {output_dir}/CFL_MAE_Comparison_DEPT_{dept}.png")

Saved: DEPT_61_OUTPUTS/CFL_MAE_Comparison_DEPT_61.png
Saved: DEPT_69_OUTPUTS/CFL_MAE_Comparison_DEPT_69.png


In [25]:
# Génération de graphiques à barres comparant les scores R² des 4 méthodes CFL par département

for dept in DEPARTMENTS:
    output_dir = f'DEPT_{dept}_OUTPUTS'
    os.makedirs(output_dir, exist_ok=True)
    
    df_dept = df_all[df_all['dept'] == dept]
    summary_dept = df_dept.groupby('method').agg({'mse': 'mean', 'mae': 'mean', 'r2': 'mean'}).reset_index()
    summary_dept.columns = ['Method', 'MSE', 'MAE', 'R²']
    
    x_pos = np.arange(len(summary_dept))
    colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']
    
    # Visualisation du coefficient de détermination R² pour évaluer la qualité d'ajustement
    plt.figure(figsize=(10, 6))
    plt.bar(x_pos, summary_dept['R²'], color=colors, alpha=0.8, edgecolor='black')
    plt.xlabel('Method', fontsize=12)
    plt.ylabel('R² Score', fontsize=12)
    plt.title(f'DEPT {dept} - CFL Methods Comparison (R²)', fontsize=14, fontweight='bold')
    plt.xticks(x_pos, summary_dept['Method'], rotation=0)
    plt.grid(axis='y', alpha=0.3)
    plt.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='Baseline')
    for i, v in enumerate(summary_dept['R²']):
        plt.text(i, v + 0.01, f"{v:.4f}", ha='center', va='bottom', fontsize=10)
    plt.tight_layout()
    plt.savefig(f'{output_dir}/CFL_R2_Comparison_DEPT_{dept}.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {output_dir}/CFL_R2_Comparison_DEPT_{dept}.png")

Saved: DEPT_61_OUTPUTS/CFL_R2_Comparison_DEPT_61.png
Saved: DEPT_69_OUTPUTS/CFL_R2_Comparison_DEPT_69.png


In [26]:
# Analyse de la variance et de la stabilité des méthodes CFL (écart-type MSE et percentiles)

for dept in DEPARTMENTS:
    output_dir = f'DEPT_{dept}_OUTPUTS'
    os.makedirs(output_dir, exist_ok=True)
    
    df_dept = df_all[df_all['dept'] == dept]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    methods = ['CFL-1', 'CFL-2', 'CFL-3', 'CFL-4']
    mse_stds = [df_dept[df_dept['method'] == m]['mse'].std() for m in methods]
    mse_means = [df_dept[df_dept['method'] == m]['mse'].mean() for m in methods]
    
    x_pos_var = np.arange(len(methods))
    
    # Graphique 1: Écart-type MSE (mesure de stabilité - plus faible = plus consistant)
    axes[0].bar(x_pos_var, mse_stds, color=['#3498db', '#e74c3c', '#2ecc71', '#f39c12'], alpha=0.8, edgecolor='black')
    axes[0].set_xlabel('Method', fontsize=12)
    axes[0].set_ylabel('MSE Standard Deviation', fontsize=12)
    axes[0].set_title(f'DEPT {dept} - CFL Stability: Lower σ = More Consistent', fontsize=14, fontweight='bold')
    axes[0].set_xticks(x_pos_var)
    axes[0].set_xticklabels(methods)
    axes[0].grid(axis='y', alpha=0.3)
    for i, v in enumerate(mse_stds):
        axes[0].text(i, v + 0.01, f"σ={v:.3f}", ha='center', va='bottom', fontsize=10)
    
    # Graphique 2: Comparaison moyenne vs 90e percentile (protection des cas défavorables)
    mse_90th = [df_dept[df_dept['method'] == m]['mse'].quantile(0.90) for m in methods]
    mse_worst = [df_dept[df_dept['method'] == m]['mse'].max() for m in methods]
    
    axes[1].bar(x_pos_var - 0.2, mse_means, width=0.4, label='Mean MSE', color='lightblue', alpha=0.7, edgecolor='black')
    axes[1].bar(x_pos_var + 0.2, mse_90th, width=0.4, label='90th Percentile', color='orange', alpha=0.7, edgecolor='black')
    axes[1].axhline(y=np.mean(mse_worst), color='red', linestyle='--', alpha=0.5, label=f'Max MSE: {np.mean(mse_worst):.1f}')
    axes[1].set_xlabel('Method', fontsize=12)
    axes[1].set_ylabel('MSE', fontsize=12)
    axes[1].set_title(f'DEPT {dept} - CFL Protects Worst-Case Performance', fontsize=14, fontweight='bold')
    axes[1].set_xticks(x_pos_var)
    axes[1].set_xticklabels(methods)
    axes[1].legend(fontsize=10)
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}/CFL_Variance_Analysis_DEPT_{dept}.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {output_dir}/CFL_Variance_Analysis_DEPT_{dept}.png")

Saved: DEPT_61_OUTPUTS/CFL_Variance_Analysis_DEPT_61.png
Saved: DEPT_69_OUTPUTS/CFL_Variance_Analysis_DEPT_69.png


In [27]:
# Analyse de la distribution des métriques avec des boxplots (MSE, MAE, R²) par méthode

for dept in DEPARTMENTS:
    output_dir = f'DEPT_{dept}_OUTPUTS'
    os.makedirs(output_dir, exist_ok=True)
    
    df_dept = df_all[df_all['dept'] == dept]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f'DEPT {dept} - Distribution Analysis by Method', fontsize=16, fontweight='bold')
    
    # Boxplot MSE : affichage de la médiane, quartiles et valeurs extrêmes
    df_dept.boxplot(column='mse', by='method', ax=axes[0], grid=False)
    axes[0].set_title('MSE Distribution')
    axes[0].set_xlabel('Method')
    axes[0].set_ylabel('MSE')
    axes[0].get_figure().suptitle('')
    
    # Boxplot MAE
    df_dept.boxplot(column='mae', by='method', ax=axes[1], grid=False)
    axes[1].set_title('MAE Distribution')
    axes[1].set_xlabel('Method')
    axes[1].set_ylabel('MAE')
    axes[1].get_figure().suptitle('')
    
    # Boxplot R²
    df_dept.boxplot(column='r2', by='method', ax=axes[2], grid=False)
    axes[2].set_title('R² Distribution')
    axes[2].set_xlabel('Method')
    axes[2].set_ylabel('R²')
    axes[2].get_figure().suptitle('')
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}/CFL_Distribution_Analysis_DEPT_{dept}.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {output_dir}/CFL_Distribution_Analysis_DEPT_{dept}.png")

Saved: DEPT_61_OUTPUTS/CFL_Distribution_Analysis_DEPT_61.png
Saved: DEPT_69_OUTPUTS/CFL_Distribution_Analysis_DEPT_69.png


In [28]:
# Visualisation de la convergence de l'entraînement FL (perte par round) pour les 4 méthodes CFL

for dept in DEPARTMENTS:
    output_dir = f'DEPT_{dept}_OUTPUTS'
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'DEPT {dept} - FL Training Convergence (Train Loss vs Rounds)', fontsize=16, fontweight='bold')
    
    methods_data = [
        (CFL1_results[dept], 'CFL-1: K-Means + FedAvg', axes[0, 0]),
        (CFL2_results[dept], 'CFL-2: K-Means + FedProx', axes[0, 1]),
        (CFL3_results[dept], 'CFL-3: Hierarchical + FedAvg', axes[1, 0]),
        (CFL4_results[dept], 'CFL-4: Hierarchical + FedProx', axes[1, 1])
    ]
    
    # Tracé de l'évolution de la perte d'entraînement pour chaque cluster
    for results_dict, title, ax in methods_data:
        for cluster_id, result in results_dict.items():
            if 'round_metrics' in result:
                df_rounds = pd.DataFrame(result['round_metrics'])
                ax.plot(df_rounds['round'], df_rounds['train_loss'], marker='o', label=f'Cluster {cluster_id}', alpha=0.7)
        
        ax.set_xlabel('Round', fontsize=11)
        ax.set_ylabel('Training Loss (MSE)', fontsize=11)
        ax.set_title(title, fontsize=12)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}/CFL_Training_Evolution_DEPT_{dept}.png', dpi=300, bbox_inches='tight')
    print(f"Saved: {output_dir}/CFL_Training_Evolution_DEPT_{dept}.png")
    plt.close()

Saved: DEPT_61_OUTPUTS/CFL_Training_Evolution_DEPT_61.png
Saved: DEPT_69_OUTPUTS/CFL_Training_Evolution_DEPT_69.png


In [29]:
# Analyse des performances MSE par cluster pour chaque méthode CFL

for dept in DEPARTMENTS:
    output_dir = f'DEPT_{dept}_OUTPUTS'
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'DEPT {dept} - Per-Cluster MSE Analysis', fontsize=16, fontweight='bold')
    
    methods_data = [
        (CFL1_results[dept], 'CFL-1: K-Means + FedAvg', axes[0, 0]),
        (CFL2_results[dept], 'CFL-2: K-Means + FedProx', axes[0, 1]),
        (CFL3_results[dept], 'CFL-3: Hierarchical + FedAvg', axes[1, 0]),
        (CFL4_results[dept], 'CFL-4: Hierarchical + FedProx', axes[1, 1])
    ]
    
    # Visualisation du MSE moyen de chaque cluster pour identifier les clusters performants
    for results_dict, title, ax in methods_data:
        cluster_ids = []
        cluster_mses = []
        
        for cluster_id, result in results_dict.items():
            df_res = pd.DataFrame(result['results'])
            cluster_ids.append(f"C{cluster_id}")
            cluster_mses.append(df_res['mse'].mean())
        
        if cluster_ids:
            ax.bar(cluster_ids, cluster_mses, color='steelblue', alpha=0.7, edgecolor='black')
            ax.set_xlabel('Cluster ID', fontsize=11)
            ax.set_ylabel('Average MSE', fontsize=11)
            ax.set_title(title, fontsize=12)
            ax.grid(axis='y', alpha=0.3)
            
            for i, v in enumerate(cluster_mses):
                ax.text(i, v + 0.02, f"{v:.3f}", ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}/CFL_Per_Cluster_Analysis_DEPT_{dept}.png', dpi=300, bbox_inches='tight')
    print(f"Saved: {output_dir}/CFL_Per_Cluster_Analysis_DEPT_{dept}.png")
    plt.close()

Saved: DEPT_61_OUTPUTS/CFL_Per_Cluster_Analysis_DEPT_61.png
Saved: DEPT_69_OUTPUTS/CFL_Per_Cluster_Analysis_DEPT_69.png


In [30]:
# Chargement des résultats des travaux pratiques précédents (TP02, TP03, TP04) pour comparaison

comparison_results = []
has_previous_tp_data = False

for dept in DEPARTMENTS:
    tp02_file = f"local_results_DEPT_{dept}_TP02.csv"
    tp03_file = f"results_per_station_global_model_DEPT_{dept}.csv"
    tp04_file = f"federated_results_DEPT_{dept}.csv"
    
    tp02_exists = os.path.exists(tp02_file)
    tp03_exists = os.path.exists(tp03_file)
    tp04_exists = os.path.exists(tp04_file)
    
    # Chargement des résultats TP02 (modèles locaux)
    if tp02_exists:
        try:
            df_tp02 = pd.read_csv(tp02_file)
            if 'mse_lr' in df_tp02.columns:
                tp02_mse = df_tp02['mse_lr'].mean()
                comparison_results.append({'Method': 'TP02-Local-LR', 'Dept': dept, 'MSE': tp02_mse})
                has_previous_tp_data = True
        except Exception as e:
            print(f"Warning: Could not read {tp02_file}: {e}")
    
    # Chargement des résultats TP03 (modèle centralisé)
    if tp03_exists:
        try:
            df_tp03 = pd.read_csv(tp03_file)
            if 'mse_lr_global' in df_tp03.columns:
                tp03_mse = df_tp03['mse_lr_global'].mean()
                comparison_results.append({'Method': 'TP03-Centralized-LR', 'Dept': dept, 'MSE': tp03_mse})
                has_previous_tp_data = True
        except Exception as e:
            print(f"Warning: Could not read {tp03_file}: {e}")
    
    # Chargement des résultats TP04 (FL global avec FedAvg et FedProx)
    if tp04_exists:
        try:
            df_tp04 = pd.read_csv(tp04_file)
            if 'mse_lr_fedavg' in df_tp04.columns:
                tp04_mse_avg = df_tp04['mse_lr_fedavg'].mean()
                comparison_results.append({'Method': 'TP04-FedAvg', 'Dept': dept, 'MSE': tp04_mse_avg})
                has_previous_tp_data = True
            if 'mse_lr_fedprox' in df_tp04.columns:
                tp04_mse_prox = df_tp04['mse_lr_fedprox'].mean()
                comparison_results.append({'Method': 'TP04-FedProx', 'Dept': dept, 'MSE': tp04_mse_prox})
                has_previous_tp_data = True
        except Exception as e:
            print(f"Warning: Could not read {tp04_file}: {e}")

In [31]:
# Ajout des résultats CFL (TP06) à la liste de comparaison

for method in ['CFL-1', 'CFL-2', 'CFL-3', 'CFL-4']:
    for dept in DEPARTMENTS:
        mse_val = df_all[(df_all['method'] == method) & (df_all['dept'] == dept)]['mse'].mean()
        if not np.isnan(mse_val):
            comparison_results.append({'Method': method, 'Dept': dept, 'MSE': mse_val})

In [32]:
# Affichage du tableau de comparaison entre tous les TP (TP02, TP03, TP04, TP06)

if has_previous_tp_data and comparison_results:
    df_comparison = pd.DataFrame(comparison_results)
    
    print("COMPARISON WITH PREVIOUS TPs (TP02, TP03, TP04, TP06)")
    print(df_comparison.to_string(index=False))

COMPARISON WITH PREVIOUS TPs (TP02, TP03, TP04, TP06)
             Method  Dept      MSE
      TP02-Local-LR    61 0.805779
TP03-Centralized-LR    61 0.780157
        TP04-FedAvg    61 0.813778
       TP04-FedProx    61 0.814080
      TP02-Local-LR    69 0.965174
TP03-Centralized-LR    69 1.010251
        TP04-FedAvg    69 0.986982
       TP04-FedProx    69 0.987173
              CFL-1    61 0.664868
              CFL-1    69 0.854012
              CFL-2    61 0.665627
              CFL-2    69 0.855039
              CFL-3    61 0.687204
              CFL-3    69 0.846387
              CFL-4    61 0.687980
              CFL-4    69 0.847422


In [33]:
# Visualisations graphiques de la comparaison MSE entre tous les TP (barres et évolution)

if has_previous_tp_data and comparison_results:
    method_order = ['TP02-Local-LR', 'TP03-Centralized-LR', 'TP04-FedAvg', 'TP04-FedProx', 
                   'CFL-1', 'CFL-2', 'CFL-3', 'CFL-4']
    method_colors = ['#95a5a6', '#3498db', '#e74c3c', '#c0392b', 
                    '#9b59b6', '#8e44ad', '#1abc9c', '#16a085']
    
    # Positionnement des méthodes sur l'axe temporel des TP
    tp_progression = {
        'TP02-Local-LR': 1,
        'TP03-Centralized-LR': 2,
        'TP04-FedAvg': 3,
        'TP04-FedProx': 3.2,
        'CFL-1': 4,
        'CFL-2': 4.2,
        'CFL-3': 4.4,
        'CFL-4': 4.6
    }
    
    for dept in DEPARTMENTS:
        output_dir = f'DEPT_{dept}_OUTPUTS'
        df_dept_comp = df_comparison[df_comparison['Dept'] == dept].copy()
        
        df_dept_comp['Method'] = pd.Categorical(df_dept_comp['Method'], categories=method_order, ordered=True)
        df_dept_comp = df_dept_comp.sort_values('Method').dropna(subset=['Method'])
        
        colors_filtered = [method_colors[method_order.index(m)] for m in df_dept_comp['Method']]
        
        # Graphique 1 : Barres comparant les MSE de toutes les méthodes
        plt.figure(figsize=(10, 6))
        plt.bar(range(len(df_dept_comp)), df_dept_comp['MSE'], 
                     color=colors_filtered, alpha=0.8, edgecolor='black', width=0.6)
        plt.xlabel('Method', fontsize=12)
        plt.ylabel('MSE', fontsize=12)
        plt.title(f'DEPT {dept} - MSE Comparison: TP02 (Local) to TP06 (Clustered FL)', fontsize=13, fontweight='bold')
        plt.xticks(range(len(df_dept_comp)), df_dept_comp['Method'], rotation=45, ha='right', fontsize=9)
        plt.grid(axis='y', alpha=0.3)
        
        for i, v in enumerate(df_dept_comp['MSE']):
            plt.text(i, v + 0.02, f"{v:.3f}", ha='center', va='bottom', fontsize=9)
        
        # Mise en évidence de la meilleure méthode
        best_idx = df_dept_comp['MSE'].idxmin()
        best_pos = df_dept_comp.index.get_loc(best_idx)
        plt.bar(best_pos, df_dept_comp.loc[best_idx, 'MSE'], 
                     color='gold', alpha=0.9, edgecolor='darkgoldenrod', linewidth=2, width=0.6)
        
        plt.tight_layout()
        plt.savefig(f'{output_dir}/TP_Comparison_Bar_DEPT_{dept}.png', dpi=300, bbox_inches='tight')
        print(f"Saved: {output_dir}/TP_Comparison_Bar_DEPT_{dept}.png")
        plt.close()
        
        # Graphique 2 : Évolution du MSE à travers les différents TP
        df_dept_comp['x_pos'] = df_dept_comp['Method'].map(tp_progression)
        df_dept_comp = df_dept_comp.dropna(subset=['x_pos']).sort_values('x_pos')
        
        tp02_03 = df_dept_comp[df_dept_comp['Method'].str.contains('TP02|TP03')]
        tp04 = df_dept_comp[df_dept_comp['Method'].str.contains('TP04')]
        cfl = df_dept_comp[df_dept_comp['Method'].str.contains('CFL')]
        
        plt.figure(figsize=(10, 6))
        if len(tp02_03) > 0:
            plt.plot(tp02_03['x_pos'], tp02_03['MSE'], marker='o', markersize=8, 
                          linewidth=2, label='Local/Centralized (TP02-03)', color='#3498db', linestyle='--')
        if len(tp04) > 0:
            plt.plot(tp04['x_pos'], tp04['MSE'], marker='s', markersize=8, 
                          linewidth=2, label='Global FL (TP04)', color='#e74c3c', linestyle='-.')
        if len(cfl) > 0:
            plt.plot(cfl['x_pos'], cfl['MSE'], marker='^', markersize=8, 
                          linewidth=2.5, label='Clustered FL (TP06)', color='#2ecc71', linestyle='-')
        
        plt.xlabel('TP Progression', fontsize=12)
        plt.ylabel('MSE', fontsize=12)
        plt.title(f'DEPT {dept} - MSE Evolution: Local to Clustered FL', fontsize=13, fontweight='bold')
        plt.legend(fontsize=10, loc='best')
        plt.grid(alpha=0.3)
        
        # Annotation de la meilleure méthode CFL
        if len(cfl) > 0:
            best_cfl = cfl.loc[cfl['MSE'].idxmin()]
            plt.annotate(f'Best: {best_cfl["Method"]}\nMSE={best_cfl["MSE"]:.3f}',
                             xy=(best_cfl['x_pos'], best_cfl['MSE']),
                             xytext=(best_cfl['x_pos'] - 0.3, best_cfl['MSE'] + 0.15),
                             arrowprops=dict(arrowstyle='->', color='green', lw=1.5),
                             fontsize=9, color='darkgreen', fontweight='bold',
                             bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', alpha=0.7))
        
        plt.tight_layout()
        plt.savefig(f'{output_dir}/TP_Comparison_Evolution_DEPT_{dept}.png', dpi=300, bbox_inches='tight')
        print(f"Saved: {output_dir}/TP_Comparison_Evolution_DEPT_{dept}.png")
        plt.close()

Saved: DEPT_61_OUTPUTS/TP_Comparison_Bar_DEPT_61.png
Saved: DEPT_61_OUTPUTS/TP_Comparison_Evolution_DEPT_61.png
Saved: DEPT_69_OUTPUTS/TP_Comparison_Bar_DEPT_69.png
Saved: DEPT_69_OUTPUTS/TP_Comparison_Evolution_DEPT_69.png


In [34]:
# Analyse statistique comparative : calcul des améliorations CFL par rapport aux approches précédentes

if has_previous_tp_data and comparison_results:
    print("STATISTICAL COMPARISON: CFL (TP06) vs Other Approaches")
    
    for dept in DEPARTMENTS:
        print(f"\nDepartment {dept}:")
        print("-" * 60)
        df_dept_comp = df_comparison[df_comparison['Dept'] == dept]
        
        # Extraction des MSE des différentes approches
        tp02_mse = df_dept_comp[df_dept_comp['Method'] == 'TP02-Local-LR']['MSE'].values
        tp03_mse = df_dept_comp[df_dept_comp['Method'] == 'TP03-Centralized-LR']['MSE'].values
        tp04_fedavg_mse = df_dept_comp[df_dept_comp['Method'] == 'TP04-FedAvg']['MSE'].values
        
        cfl_methods = df_dept_comp[df_dept_comp['Method'].str.contains('CFL')]
        best_cfl = cfl_methods.loc[cfl_methods['MSE'].idxmin()]
        avg_cfl_mse = cfl_methods['MSE'].mean()
        
        print(f"  Best CFL Method: {best_cfl['Method']} (MSE: {best_cfl['MSE']:.4f})")
        print(f"  Average CFL MSE: {avg_cfl_mse:.4f}")
        
        # Calcul des pourcentages d'amélioration par rapport à chaque TP
        if len(tp02_mse) > 0:
            improvement_tp02 = ((tp02_mse[0] - best_cfl['MSE']) / tp02_mse[0]) * 100
            print(f"  vs TP02 (Local): {improvement_tp02:+.2f}% ({'improvement' if improvement_tp02 > 0 else 'degradation'})")
        
        if len(tp03_mse) > 0:
            improvement_tp03 = ((tp03_mse[0] - best_cfl['MSE']) / tp03_mse[0]) * 100
            gap_to_centralized = best_cfl['MSE'] - tp03_mse[0]
            print(f"  vs TP03 (Centralized): {improvement_tp03:+.2f}% ({'improvement' if improvement_tp03 > 0 else 'gap'})")
            print(f"    Gap to centralized: {gap_to_centralized:.4f} MSE ({abs(gap_to_centralized/tp03_mse[0]*100):.2f}%)")
        
        if len(tp04_fedavg_mse) > 0:
            improvement_tp04 = ((tp04_fedavg_mse[0] - best_cfl['MSE']) / tp04_fedavg_mse[0]) * 100
            print(f"  vs TP04 (Global FedAvg): {improvement_tp04:+.2f}% ({'improvement' if improvement_tp04 > 0 else 'degradation'})")

STATISTICAL COMPARISON: CFL (TP06) vs Other Approaches

Department 61:
------------------------------------------------------------
  Best CFL Method: CFL-1 (MSE: 0.6649)
  Average CFL MSE: 0.6764
  vs TP02 (Local): +17.49% (improvement)
  vs TP03 (Centralized): +14.78% (improvement)
    Gap to centralized: -0.1153 MSE (14.78%)
  vs TP04 (Global FedAvg): +18.30% (improvement)

Department 69:
------------------------------------------------------------
  Best CFL Method: CFL-3 (MSE: 0.8464)
  Average CFL MSE: 0.8507
  vs TP02 (Local): +12.31% (improvement)
  vs TP03 (Centralized): +16.22% (improvement)
    Gap to centralized: -0.1639 MSE (16.22%)
  vs TP04 (Global FedAvg): +14.24% (improvement)


In [35]:
# Synthèse finale de l'analyse CFL par département avec identification de la meilleure méthode

print("FINAL ANALYSIS SUMMARY (PER-DEPARTMENT)")

for dept in DEPARTMENTS:
    df_dept = df_all[df_all['dept'] == dept]
    summary_dept = df_dept.groupby('method').agg({
        'mse': 'mean',
        'mae': 'mean',
        'r2': 'mean'
    }).reset_index()
    
    summary_dept.columns = ['Method', 'MSE', 'MAE', 'R²']
    summary_dept['Clustering'] = ['K-means', 'K-means', 'Hierarchical', 'Hierarchical']
    summary_dept['FL'] = ['FedAvg', 'FedProx', 'FedAvg', 'FedProx']
    summary_dept = summary_dept[['Method', 'Clustering', 'FL', 'MSE', 'MAE', 'R²']]
    
    print(f"DEPARTMENT {dept} - FINAL SUMMARY")
    
    # Identification de la meilleure méthode (MSE le plus faible)
    best_method = summary_dept.loc[summary_dept['MSE'].idxmin()]
    print(f"\nBest Method (Lowest MSE): {best_method['Method']}")
    print(f"  Clustering: {best_method['Clustering']}")
    print(f"  FL Algorithm: {best_method['FL']}")
    print(f"  MSE: {best_method['MSE']:.4f} (RMSE: {np.sqrt(best_method['MSE']):.4f}°C)")
    print(f"  MAE: {best_method['MAE']:.4f}°C")
    print(f"  R²: {best_method['R²']:.4f}")
    
    # Comparaison des méthodes de clustering
    kmeans_avg = summary_dept[summary_dept['Clustering'] == 'K-means']['MSE'].mean()
    hierarchical_avg = summary_dept[summary_dept['Clustering'] == 'Hierarchical']['MSE'].mean()
    print(f"\nClustering Method Comparison:")
    print(f"  K-means Average MSE: {kmeans_avg:.4f}")
    print(f"  Hierarchical Average MSE: {hierarchical_avg:.4f}")
    print(f"  Better Clustering: {'K-means' if kmeans_avg < hierarchical_avg else 'Hierarchical'}")
    
    # Comparaison des algorithmes FL
    fedavg_avg = summary_dept[summary_dept['FL'] == 'FedAvg']['MSE'].mean()
    fedprox_avg = summary_dept[summary_dept['FL'] == 'FedProx']['MSE'].mean()
    print(f"\nFL Algorithm Comparison:")
    print(f"  FedAvg Average MSE: {fedavg_avg:.4f}")
    print(f"  FedProx Average MSE: {fedprox_avg:.4f}")
    print(f"  Better FL Algorithm: {'FedAvg' if fedavg_avg < fedprox_avg else 'FedProx'}")

FINAL ANALYSIS SUMMARY (PER-DEPARTMENT)
DEPARTMENT 61 - FINAL SUMMARY

Best Method (Lowest MSE): CFL-1
  Clustering: K-means
  FL Algorithm: FedAvg
  MSE: 0.6649 (RMSE: 0.8154°C)
  MAE: 0.6325°C
  R²: 0.9811

Clustering Method Comparison:
  K-means Average MSE: 0.6652
  Hierarchical Average MSE: 0.6876
  Better Clustering: K-means

FL Algorithm Comparison:
  FedAvg Average MSE: 0.6760
  FedProx Average MSE: 0.6768
  Better FL Algorithm: FedAvg
DEPARTMENT 69 - FINAL SUMMARY

Best Method (Lowest MSE): CFL-3
  Clustering: Hierarchical
  FL Algorithm: FedAvg
  MSE: 0.8464 (RMSE: 0.9200°C)
  MAE: 0.7150°C
  R²: 0.9843

Clustering Method Comparison:
  K-means Average MSE: 0.8545
  Hierarchical Average MSE: 0.8469
  Better Clustering: Hierarchical

FL Algorithm Comparison:
  FedAvg Average MSE: 0.8502
  FedProx Average MSE: 0.8512
  Better FL Algorithm: FedAvg
